# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention, de-duplicate from GISAID

In [1]:
# Housekeeping

import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 


# Dates
start_date = "06-06-2025"
end_date = "06-06-2025"
date_range = start_date + "--" + end_date
update_date = "06-10-2025"

# Make sure you have the correct paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = downloads + "Andersen/"
saved = originals + "saved/"
temp_files = originals + "temp/"
complete_files = downloads + "complete/"

os.chdir(downloads)

## Read Metadata 

In [2]:
# Read metadata

os.chdir(saved)
metadata_normalized = pd.read_csv("metadata_normalized.tsv", delimiter="\t") # Collection dates

metadata_folder = originals + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

print(len(metadata)) # 7397 rows

metadata = metadata.merge(metadata_normalized, how="outer")
print(metadata.columns)

metadata["name_state"] = metadata["geo_loc_name"].apply(lambda x: x.split("/")[1] if len(x.split("/")) > 1 else x.split("/")[0])

# Find only >= last date using Release Date from metadata 
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["ReleaseDate"] >= dateutil.parser.parse(start_date).strftime("%Y-%m-%d")]
# Find only <= update date using Release Date from metadata
metadata = metadata[metadata["ReleaseDate"] <= dateutil.parser.parse(end_date).strftime("%Y-%m-%d")]

print(len(metadata)) # 6053 rows between 1/1/2024 and 4/14/2025
display(metadata)

9697
Index(['Run', 'Assay Type', 'AvgSpotLen', 'Bases', 'BioProject', 'BioSample',
       'BioSampleModel', 'Bytes', 'Center Name', 'Collection_Date', 'Consent',
       'DATASTORE filetype', 'DATASTORE provider', 'DATASTORE region',
       'Experiment', 'geo_loc_name_country', 'geo_loc_name_country_continent',
       'geo_loc_name', 'Host', 'Instrument', 'isolate', 'Library Name',
       'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism',
       'Platform', 'ReleaseDate', 'create_date', 'version', 'Sample Name',
       'SRA Study', 'serotype', 'isolation_source', 'BioSample Accession',
       'is_retracted', 'retraction_detection_date_utc'],
      dtype='object')
117


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,create_date,version,Sample Name,SRA Study,serotype,isolation_source,BioSample Accession,is_retracted,retraction_detection_date_utc,name_state
18933,SRR33830324,WGS,147.97,74609244,PRJNA1207547,SAMN48895309,Viral,27537881,USDA-NVSL,2025,...,2025-06-04 14:09:57,1,25-013422-001,SRP557452,NaN,BLOOD SWAB,SRS25266441,False,NaN,USA
18934,SRR33830325,WGS,148.64,156899568,PRJNA1207547,SAMN48895308,Viral,62096192,USDA-NVSL,2025,...,2025-06-04 14:09:58,1,25-015561-001,SRP557452,NaN,CLOACAL/TRACHEAL SWAB POOL,SRS25266440,False,NaN,USA
18935,SRR33830326,WGS,137.62,616674,PRJNA1207547,SAMN48895307,Viral,313930,USDA-NVSL,2025,...,2025-06-04 14:09:31,1,25-013427-001,SRP557452,NaN,BLOOD SWAB,SRS25266439,False,NaN,USA
18936,SRR33830327,WGS,148.23,57621808,PRJNA1207547,SAMN48895306,Viral,21591302,USDA-NVSL,2025,...,2025-06-04 14:09:54,1,25-013424-001,SRP557452,NaN,BLOOD SWAB,SRS25266438,False,NaN,USA
18937,SRR33830328,WGS,148.93,140872576,PRJNA1207547,SAMN48895305,Viral,56890513,USDA-NVSL,2025,...,2025-06-04 14:10:14,1,25-015870-001,SRP557452,NaN,swab,SRS25266437,False,NaN,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19045,SRR33830446,WGS,148.77,95232771,PRJNA1102327,SAMN48895426,Viral,37952018,USDA-NVSL,2025,...,2025-06-04 14:24:56,1,25-015677-004,SRP503016,NaN,"MILK, BULK TANK",SRS25266464,False,NaN,USA
19046,SRR33830447,WGS,148.43,106145246,PRJNA1102327,SAMN48895425,Viral,42182759,USDA-NVSL,2025,...,2025-06-04 14:25:01,1,25-015677-003,SRP503016,NaN,"MILK, BULK TANK",SRS25266463,False,NaN,USA
19047,SRR33830448,WGS,148.30,110812426,PRJNA1102327,SAMN48895424,Viral,43738503,USDA-NVSL,2025,...,2025-06-04 14:24:55,1,25-015677-002,SRP503016,NaN,"MILK, BULK TANK",SRS25266462,False,NaN,USA
19048,SRR33830449,WGS,131.52,67358891,PRJNA1102327,SAMN48895415,Viral,26336550,USDA-NVSL,2024,...,2025-06-04 14:24:53,1,24-036379-001-tile,SRP503016,NaN,"MILK, BULK TANK",SRS25266461,False,NaN,USA


In [3]:
# Get list of genotypes

# os.chdir(home)

# genotypes_df = pd.read_excel("genotype_key.xlsx")

# genotypes = list(genotypes_df["Genotype"])

# print(genotypes)

genotypes = ["B3.13", "D1.1"] #, "B3.2", "B3.6", "B3.7", "B3.5", "A3"]

### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[collection_date]|[host_type]|[genotype]

host_type is from manual animal reference

In metadata, we have: host, geo_loc_name, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name (primary) = geo_loc_name

geo_loc_name (secondary) = genbank_mapping.tsv > genbank_name

isolate = isolate

collection date (primary) = Collection_Date

collection date (secondary) = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = serotype

host type = [from ref] 

genotype = [from genoflu] -- use genoflu_results.tsv

## Get genotype, specific geolocation

In [4]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")

genoflu_results["Run"] = genoflu_results["sample"]

metadata = metadata.merge(genoflu_results, on="Run", how="inner")
print(metadata)
# metadata = metadata[~metadata["Genotype"].str.contains('Not assigned')] # Do not include non-assigned genotypes
metadata = metadata[metadata["Genotype"].isin(genotypes)]

# Get only the genotypes we want: B3.13 and D1.1

# b313_and_d11_only = genoflu_results[(genoflu_results["Genotype"] == "B3.13") | (genoflu_results["Genotype"] == "D1.1")]
# b313_and_d11_only = b313_and_d11_only.rename(columns={"sample": "Run"})
# b313_and_d11_only = b313_and_d11_only.drop_duplicates(subset="Run", keep="last")

# metadata = metadata.merge(b313_and_d11_only, on=["Run", "Genotype"], how="inner")

print(len(metadata)) 

display(metadata)

             Run Assay Type  AvgSpotLen      Bases    BioProject  \
0    SRR33830324        WGS      147.97   74609244  PRJNA1207547   
1    SRR33830325        WGS      148.64  156899568  PRJNA1207547   
2    SRR33830326        WGS      137.62     616674  PRJNA1207547   
3    SRR33830327        WGS      148.23   57621808  PRJNA1207547   
4    SRR33830328        WGS      148.93  140872576  PRJNA1207547   
..           ...        ...         ...        ...           ...   
112  SRR33830446        WGS      148.77   95232771  PRJNA1102327   
113  SRR33830447        WGS      148.43  106145246  PRJNA1102327   
114  SRR33830448        WGS      148.30  110812426  PRJNA1102327   
115  SRR33830449        WGS      131.52   67358891  PRJNA1102327   
116  SRR33830450        WGS      131.13   69602732  PRJNA1102327   

        BioSample BioSampleModel     Bytes Center Name Collection_Date  ...  \
0    SAMN48895309          Viral  27537881   USDA-NVSL            2025  ...   
1    SAMN48895308        

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,name_state,sample,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List
0,SRR33830324,WGS,147.97,74609244,PRJNA1207547,SAMN48895309,Viral,27537881,USDA-NVSL,2025,...,USA,SRR33830324,2025-06-06_07-56-21,SRR33830324.fa,D1.1,"NP:am13, NS:ea3, PB2:am24, HA:ea3, PB1:ea3, MP...","am13:24-030039-001:NP, ea3:22-013001-001:NS, a...","99.80%, 99.17%, 99.74%, 99.24%, 99.34%, 100.00...","3, 7, 6, 13, 15, 0, 10, 7",Ran on FASTA - No Coverage Report
1,SRR33830325,WGS,148.64,156899568,PRJNA1207547,SAMN48895308,Viral,62096192,USDA-NVSL,2025,...,USA,SRR33830325,2025-06-06_07-56-21,SRR33830325.fa,D1.1,"PA:am4, PB2:am24, HA:ea3, PB1:ea3, MP:ea3, NS:...","am4:24-030039-001:PA, am24:24-030039-001:PB2, ...","99.60%, 99.65%, 99.41%, 99.16%, 99.90%, 98.93%...","7, 8, 10, 19, 1, 9, 7, 2",Ran on FASTA - No Coverage Report
2,SRR33830326,WGS,137.62,616674,PRJNA1207547,SAMN48895307,Viral,313930,USDA-NVSL,2025,...,USA,SRR33830326,2025-06-06_07-56-21,SRR33830326.fa,D1.1,"MP:ea3, PB1:ea3, NA:am4N1, PB2:am24, HA:ea3, N...","ea3:22-013001-001:MP, ea3:22-013001-001:PB1, a...","99.90%, 99.68%, 99.33%, 99.78%, 99.41%, 98.93%...","1, 2, 7, 5, 10, 9, 2, 5",Ran on FASTA - No Coverage Report
3,SRR33830327,WGS,148.23,57621808,PRJNA1207547,SAMN48895306,Viral,21591302,USDA-NVSL,2025,...,USA,SRR33830327,2025-06-06_07-56-20,SRR33830327.fa,D1.1,"PA:am4, MP:ea3, NP:am13, NS:ea3, NA:am4N1, PB2...","am4:24-030039-001:PA, ea3:22-013001-001:MP, am...","99.81%, 99.90%, 99.87%, 99.28%, 99.52%, 99.69%...","4, 1, 2, 6, 5, 7, 20, 11",Ran on FASTA - No Coverage Report
4,SRR33830328,WGS,148.93,140872576,PRJNA1207547,SAMN48895305,Viral,56890513,USDA-NVSL,2025,...,USA,SRR33830328,2025-06-06_07-56-20,SRR33830328.fa,D1.1,"NS:ea3, NA:am4N1, PB1:ea3, MP:ea3, PB2:am24, H...","ea3:22-013001-001:NS, am4N1:24-030039-001:NA, ...","99.17%, 99.03%, 99.30%, 99.90%, 99.61%, 99.41%...","7, 11, 16, 1, 9, 10, 13, 5",Ran on FASTA - No Coverage Report
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,SRR33830443,WGS,148.11,100597029,PRJNA1102327,SAMN48895429,Viral,39710063,USDA-NVSL,2025,...,USA,SRR33830443,2025-06-06_07-56-21,SRR33830443.fa,B3.13,"NA:ea1, PB1:am4, MP:ea1, PA:ea1, NP:am8, HA:ea...","ea1:22-003707-003:NA, am4:23-001855-001:PB1, e...","98.72%, 99.52%, 98.98%, 98.88%, 98.86%, 98.06%...","18, 11, 10, 24, 17, 33, 38, 9",Ran on FASTA - No Coverage Report
110,SRR33830444,WGS,148.50,131987248,PRJNA1102327,SAMN48895428,Viral,51617990,USDA-NVSL,2025,...,USA,SRR33830444,2025-06-06_07-56-20,SRR33830444.fa,B3.13,"PA:ea1, NA:ea1, PB2:am2.2, NP:am8, HA:ea1, MP:...","ea1:22-003707-003:PA, ea1:22-003707-003:NA, am...","98.88%, 98.65%, 98.20%, 98.93%, 98.06%, 98.98%...","24, 19, 41, 16, 33, 10, 12, 9",Ran on FASTA - No Coverage Report
111,SRR33830445,WGS,148.12,115677479,PRJNA1102327,SAMN48895427,Viral,45674405,USDA-NVSL,2025,...,USA,SRR33830445,2025-06-06_07-56-21,SRR33830445.fa,B3.13,"NP:am8, HA:ea1, PA:ea1, NS:am1.1, PB1:am4, PB2...","am8:23-032005-001:NP, ea1:22-003707-003:HA, ea...","98.93%, 98.24%, 98.79%, 98.93%, 99.56%, 98.51%...","16, 30, 26, 9, 10, 34, 19, 11",Ran on FASTA - No Coverage Report
115,SRR33830449,WGS,131.52,67358891,PRJNA1102327,SAMN48895415,Viral,26336550,USDA-NVSL,2024,...,USA,SRR33830449,2025-06-06_07-56-20,SRR33830449.fa,B3.13,"NP:am8, NA:ea1, PB1:am4, HA:ea1, NS:am1.1, MP:...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","98.93%, 98.86%, 99.47%, 98.43%, 99.17%, 98.95%...","16, 16, 12, 12, 7, 10, 31, 24",Ran on FASTA - No Coverage Report


In [5]:
# Get specific geolocation and name_state from genbank_mapping.tsv

genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping["Run"] = genbank_mapping["sra_run"]
genbank_mapping = genbank_mapping.drop_duplicates(subset="Run", keep="first") # Drop duplicates
genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2]) # Get the name of the state

# print(genbank_mapping)

# metadata_genbank = metadata.merge(genbank_mapping, on=["Run"]) # Only include data that has states

# Get geolocation for second state attribute

os.chdir(home + "references/")
state_ref = pd.read_csv("states_ref.csv")
metadata["Geo_Location"] = metadata["name_state"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"] == x, 'Country'].iloc[0] + "-" + x if x in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x, 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x, 'Abbreviation'].iloc[0] if x in state_ref["State"].values else x)


print(metadata["name_state"])
print(len(metadata))
display(metadata) # Maybe there is no state information since 3/18/2025?

0      USA
1      USA
2      USA
3      USA
4      USA
      ... 
109    USA
110    USA
111    USA
115    USA
116    USA
Name: name_state, Length: 111, dtype: object
111


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,sample,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,Geo_Location
0,SRR33830324,WGS,147.97,74609244,PRJNA1207547,SAMN48895309,Viral,27537881,USDA-NVSL,2025,...,SRR33830324,2025-06-06_07-56-21,SRR33830324.fa,D1.1,"NP:am13, NS:ea3, PB2:am24, HA:ea3, PB1:ea3, MP...","am13:24-030039-001:NP, ea3:22-013001-001:NS, a...","99.80%, 99.17%, 99.74%, 99.24%, 99.34%, 100.00...","3, 7, 6, 13, 15, 0, 10, 7",Ran on FASTA - No Coverage Report,USA
1,SRR33830325,WGS,148.64,156899568,PRJNA1207547,SAMN48895308,Viral,62096192,USDA-NVSL,2025,...,SRR33830325,2025-06-06_07-56-21,SRR33830325.fa,D1.1,"PA:am4, PB2:am24, HA:ea3, PB1:ea3, MP:ea3, NS:...","am4:24-030039-001:PA, am24:24-030039-001:PB2, ...","99.60%, 99.65%, 99.41%, 99.16%, 99.90%, 98.93%...","7, 8, 10, 19, 1, 9, 7, 2",Ran on FASTA - No Coverage Report,USA
2,SRR33830326,WGS,137.62,616674,PRJNA1207547,SAMN48895307,Viral,313930,USDA-NVSL,2025,...,SRR33830326,2025-06-06_07-56-21,SRR33830326.fa,D1.1,"MP:ea3, PB1:ea3, NA:am4N1, PB2:am24, HA:ea3, N...","ea3:22-013001-001:MP, ea3:22-013001-001:PB1, a...","99.90%, 99.68%, 99.33%, 99.78%, 99.41%, 98.93%...","1, 2, 7, 5, 10, 9, 2, 5",Ran on FASTA - No Coverage Report,USA
3,SRR33830327,WGS,148.23,57621808,PRJNA1207547,SAMN48895306,Viral,21591302,USDA-NVSL,2025,...,SRR33830327,2025-06-06_07-56-20,SRR33830327.fa,D1.1,"PA:am4, MP:ea3, NP:am13, NS:ea3, NA:am4N1, PB2...","am4:24-030039-001:PA, ea3:22-013001-001:MP, am...","99.81%, 99.90%, 99.87%, 99.28%, 99.52%, 99.69%...","4, 1, 2, 6, 5, 7, 20, 11",Ran on FASTA - No Coverage Report,USA
4,SRR33830328,WGS,148.93,140872576,PRJNA1207547,SAMN48895305,Viral,56890513,USDA-NVSL,2025,...,SRR33830328,2025-06-06_07-56-20,SRR33830328.fa,D1.1,"NS:ea3, NA:am4N1, PB1:ea3, MP:ea3, PB2:am24, H...","ea3:22-013001-001:NS, am4N1:24-030039-001:NA, ...","99.17%, 99.03%, 99.30%, 99.90%, 99.61%, 99.41%...","7, 11, 16, 1, 9, 10, 13, 5",Ran on FASTA - No Coverage Report,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,SRR33830443,WGS,148.11,100597029,PRJNA1102327,SAMN48895429,Viral,39710063,USDA-NVSL,2025,...,SRR33830443,2025-06-06_07-56-21,SRR33830443.fa,B3.13,"NA:ea1, PB1:am4, MP:ea1, PA:ea1, NP:am8, HA:ea...","ea1:22-003707-003:NA, am4:23-001855-001:PB1, e...","98.72%, 99.52%, 98.98%, 98.88%, 98.86%, 98.06%...","18, 11, 10, 24, 17, 33, 38, 9",Ran on FASTA - No Coverage Report,USA
110,SRR33830444,WGS,148.50,131987248,PRJNA1102327,SAMN48895428,Viral,51617990,USDA-NVSL,2025,...,SRR33830444,2025-06-06_07-56-20,SRR33830444.fa,B3.13,"PA:ea1, NA:ea1, PB2:am2.2, NP:am8, HA:ea1, MP:...","ea1:22-003707-003:PA, ea1:22-003707-003:NA, am...","98.88%, 98.65%, 98.20%, 98.93%, 98.06%, 98.98%...","24, 19, 41, 16, 33, 10, 12, 9",Ran on FASTA - No Coverage Report,USA
111,SRR33830445,WGS,148.12,115677479,PRJNA1102327,SAMN48895427,Viral,45674405,USDA-NVSL,2025,...,SRR33830445,2025-06-06_07-56-21,SRR33830445.fa,B3.13,"NP:am8, HA:ea1, PA:ea1, NS:am1.1, PB1:am4, PB2...","am8:23-032005-001:NP, ea1:22-003707-003:HA, ea...","98.93%, 98.24%, 98.79%, 98.93%, 99.56%, 98.51%...","16, 30, 26, 9, 10, 34, 19, 11",Ran on FASTA - No Coverage Report,USA
115,SRR33830449,WGS,131.52,67358891,PRJNA1102327,SAMN48895415,Viral,26336550,USDA-NVSL,2024,...,SRR33830449,2025-06-06_07-56-20,SRR33830449.fa,B3.13,"NP:am8, NA:ea1, PB1:am4, HA:ea1, NS:am1.1, MP:...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","98.93%, 98.86%, 99.47%, 98.43%, 99.17%, 98.95%...","16, 16, 12, 12, 7, 10, 31, 24",Ran on FASTA - No Coverage Report,USA


In [6]:
# # If no states

# metadata_genbank = metadata

# metadata_genbank["name_state"] = "USA"

# metadata_genbank["Geo_Location"] = "USA"

# display(metadata_genbank)

## Collection Dates

If date is N/A, try finding it first

In [ ]:

# # Get all dates
# metadata["Collection_Date_Specific"] = metadata["BioSample"].apply(lambda x: search_collection_date(x, metadata) if "-" not in x else x)

# # Save this so we don't have to do it again

# # os.chdir(temp_files)
# metadata.to_csv("metadata_genbank.csv")

Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable t

In [9]:
# # Upload saved data -- if doing this, make sure the above cell is commented out
# os.chdir(temp_files + "saved/")
# metadata_genbank = pd.read_csv("metadata_genbank.csv")
# os.chdir(temp_files)

# Get only updated dates

unknown_dates = metadata[(metadata["Collection_Date"] == "2024") | (metadata["Collection_Date"] == "2025")] # Dates we don't have
known_dates = metadata[(metadata["Collection_Date"] != "2024") & (metadata["Collection_Date"] != "2025")] # Dates we've already gotten

# Get new dates also 
# new_dates = metadata["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank) if )

updated_unknown_dates = unknown_dates["BioSample"].apply(lambda x: search_collection_date(x, unknown_dates)) # Update unknown dates, if possible
unknown_dates["Collection_Date"] = updated_unknown_dates

metadata_genbank = pd.concat([known_dates, unknown_dates], ignore_index=True, sort=True)

# # Remove pre-2024 dates

# metadata_genbank["Collection_Date_Compare"] = metadata_genbank["Collection_Date_Specific"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
# metadata_genbank = metadata_genbank[metadata_genbank["Collection_Date_Compare"] >= datetime(2024, 1, 1).strftime("%Y-%m-%d")]

# print(metadata[["Collection_Date_Specific"]])

display(metadata)

Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable t

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,Geo_Location,Collection_Date_Specific
0,SRR33830324,WGS,147.97,74609244,PRJNA1207547,SAMN48895309,Viral,27537881,USDA-NVSL,2025,...,2025-06-06_07-56-21,SRR33830324.fa,D1.1,"NP:am13, NS:ea3, PB2:am24, HA:ea3, PB1:ea3, MP...","am13:24-030039-001:NP, ea3:22-013001-001:NS, a...","99.80%, 99.17%, 99.74%, 99.24%, 99.34%, 100.00...","3, 7, 6, 13, 15, 0, 10, 7",Ran on FASTA - No Coverage Report,USA,2025
1,SRR33830325,WGS,148.64,156899568,PRJNA1207547,SAMN48895308,Viral,62096192,USDA-NVSL,2025,...,2025-06-06_07-56-21,SRR33830325.fa,D1.1,"PA:am4, PB2:am24, HA:ea3, PB1:ea3, MP:ea3, NS:...","am4:24-030039-001:PA, am24:24-030039-001:PB2, ...","99.60%, 99.65%, 99.41%, 99.16%, 99.90%, 98.93%...","7, 8, 10, 19, 1, 9, 7, 2",Ran on FASTA - No Coverage Report,USA,2025
2,SRR33830326,WGS,137.62,616674,PRJNA1207547,SAMN48895307,Viral,313930,USDA-NVSL,2025,...,2025-06-06_07-56-21,SRR33830326.fa,D1.1,"MP:ea3, PB1:ea3, NA:am4N1, PB2:am24, HA:ea3, N...","ea3:22-013001-001:MP, ea3:22-013001-001:PB1, a...","99.90%, 99.68%, 99.33%, 99.78%, 99.41%, 98.93%...","1, 2, 7, 5, 10, 9, 2, 5",Ran on FASTA - No Coverage Report,USA,2025
3,SRR33830327,WGS,148.23,57621808,PRJNA1207547,SAMN48895306,Viral,21591302,USDA-NVSL,2025,...,2025-06-06_07-56-20,SRR33830327.fa,D1.1,"PA:am4, MP:ea3, NP:am13, NS:ea3, NA:am4N1, PB2...","am4:24-030039-001:PA, ea3:22-013001-001:MP, am...","99.81%, 99.90%, 99.87%, 99.28%, 99.52%, 99.69%...","4, 1, 2, 6, 5, 7, 20, 11",Ran on FASTA - No Coverage Report,USA,2025
4,SRR33830328,WGS,148.93,140872576,PRJNA1207547,SAMN48895305,Viral,56890513,USDA-NVSL,2025,...,2025-06-06_07-56-20,SRR33830328.fa,D1.1,"NS:ea3, NA:am4N1, PB1:ea3, MP:ea3, PB2:am24, H...","ea3:22-013001-001:NS, am4N1:24-030039-001:NA, ...","99.17%, 99.03%, 99.30%, 99.90%, 99.61%, 99.41%...","7, 11, 16, 1, 9, 10, 13, 5",Ran on FASTA - No Coverage Report,USA,2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,SRR33830443,WGS,148.11,100597029,PRJNA1102327,SAMN48895429,Viral,39710063,USDA-NVSL,2025,...,2025-06-06_07-56-21,SRR33830443.fa,B3.13,"NA:ea1, PB1:am4, MP:ea1, PA:ea1, NP:am8, HA:ea...","ea1:22-003707-003:NA, am4:23-001855-001:PB1, e...","98.72%, 99.52%, 98.98%, 98.88%, 98.86%, 98.06%...","18, 11, 10, 24, 17, 33, 38, 9",Ran on FASTA - No Coverage Report,USA,2025
110,SRR33830444,WGS,148.50,131987248,PRJNA1102327,SAMN48895428,Viral,51617990,USDA-NVSL,2025,...,2025-06-06_07-56-20,SRR33830444.fa,B3.13,"PA:ea1, NA:ea1, PB2:am2.2, NP:am8, HA:ea1, MP:...","ea1:22-003707-003:PA, ea1:22-003707-003:NA, am...","98.88%, 98.65%, 98.20%, 98.93%, 98.06%, 98.98%...","24, 19, 41, 16, 33, 10, 12, 9",Ran on FASTA - No Coverage Report,USA,2025
111,SRR33830445,WGS,148.12,115677479,PRJNA1102327,SAMN48895427,Viral,45674405,USDA-NVSL,2025,...,2025-06-06_07-56-21,SRR33830445.fa,B3.13,"NP:am8, HA:ea1, PA:ea1, NS:am1.1, PB1:am4, PB2...","am8:23-032005-001:NP, ea1:22-003707-003:HA, ea...","98.93%, 98.24%, 98.79%, 98.93%, 99.56%, 98.51%...","16, 30, 26, 9, 10, 34, 19, 11",Ran on FASTA - No Coverage Report,USA,2025
115,SRR33830449,WGS,131.52,67358891,PRJNA1102327,SAMN48895415,Viral,26336550,USDA-NVSL,2024,...,2025-06-06_07-56-20,SRR33830449.fa,B3.13,"NP:am8, NA:ea1, PB1:am4, HA:ea1, NS:am1.1, MP:...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","98.93%, 98.86%, 99.47%, 98.43%, 99.17%, 98.95%...","16, 16, 12, 12, 7, 10, 31, 24",Ran on FASTA - No Coverage Report,USA,2024


In [ ]:
# If no collection dates

# metadata_genbank["Collection_Date_Specific"] = metadata_genbank["Collection_Date"]

## Get host type

In [10]:
# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(downloads)

animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['red-tailed hawk', 'turkey vulture', 'black vulture', 'common grackle', 'chicken', 'cattle', 'bald eagle', 'great horned owl']
[]
                avian               cattle        feline   other_mammal  \
0    great_horned_owl            dairy_cow           cat     deer mouse   
1        common_raven               cattle  domestic_cat    house_mouse   
2       cooper's_hawk  cattle milk product     feral_cat          skunk   
3        coopers_hawk          bovine_milk        feline  striped_skunk   
4             peafowl              bovine   domestic-cat     norway rat   
..                ...                  ...           ...            ...   
415         gyrfalcon                  NaN           NaN            NaN   
416  american_goshawk                  NaN           NaN            NaN   
417      king_vulture                  NaN           NaN            NaN   
418     american coot                  NaN           NaN            NaN   
419        barred owl                  NaN  

In [11]:
# Get animals from animal reference
os.chdir(downloads)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata, animals_ref) # Get host type

metadata["years"] = metadata["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date

## Make names using all the attributes we collected

In [12]:
for num, collection_date in enumerate(metadata["Collection_Date"]):
    
    if collection_date != collection_date: # If nan
        metadata.loc[num, "Collection_Date"] = metadata.loc[num, "years"]
    else: # If actual date
        if len(str(collection_date)) == 4: # If it's a year
            # print("caught")
            metadata.loc[num, "Collection_Date"] = collection_date
        else:
            parsed_date = dateutil.parser.parse(collection_date)
            date = parsed_date.strftime("%Y-%m-%d") # Make sure it doesn't default to today, if just a year
            metadata.loc[num, "Collection_Date"] = date

    metadata = metadata.dropna(thresh=2)

# Make names

# + metadata["BioSample"] + "|" 
names = ">" + metadata["Run"] + "|" + "A/" + metadata["Host"] + "/" + metadata["name_state"] + "/" + metadata["isolate"] + "/" + metadata["years"].apply(lambda x: str(x)) + "|H5N1|" + metadata["Geo_Location"] + "|" + metadata["Collection_Date"].apply(lambda x: str(x)) + "|" + metadata["Host_Type"] + "|" + metadata["Genotype"]

metadata["Name"] = names

# metadata_genbank.to_csv("metadata_genbank_named.csv")

display(metadata[["ReleaseDate", 'create_date', 'Collection_Date']])

,ReleaseDate,create_date,Collection_Date
0,2025-06-06,2025-06-04 14:09:57,2025
1,2025-06-06,2025-06-04 14:09:58,2025
2,2025-06-06,2025-06-04 14:09:31,2025
3,2025-06-06,2025-06-04 14:09:54,2025
4,2025-06-06,2025-06-04 14:10:14,2025
...,...,...,...
109,2025-06-06,2025-06-04 14:25:00,2024
110,2025-06-06,2025-06-04 14:25:01,2024
111,2025-06-06,2025-06-04 14:24:54,2025
115,2025-06-06,2025-06-04 14:24:53,2024


In [13]:
# Drop duplicate runs 
metadata = metadata.drop_duplicates(subset="Run", keep="first")

In [14]:
print(metadata)

             Run Assay Type  AvgSpotLen        Bases    BioProject  \
0    SRR33830324        WGS      147.97   74609244.0  PRJNA1207547   
1    SRR33830325        WGS      148.64  156899568.0  PRJNA1207547   
2    SRR33830326        WGS      137.62     616674.0  PRJNA1207547   
3    SRR33830327        WGS      148.23   57621808.0  PRJNA1207547   
4    SRR33830328        WGS      148.93  140872576.0  PRJNA1207547   
..           ...        ...         ...          ...           ...   
109  SRR33830443        WGS      148.11  100597029.0  PRJNA1102327   
110  SRR33830444        WGS      148.50  131987248.0  PRJNA1102327   
111  SRR33830445        WGS      148.12  115677479.0  PRJNA1102327   
115  SRR33830449        WGS      131.52   67358891.0  PRJNA1102327   
116  SRR33830450        WGS      131.13   69602732.0  PRJNA1102327   

        BioSample BioSampleModel       Bytes Center Name Collection_Date  ...  \
0    SAMN48895309          Viral  27537881.0   USDA-NVSL            2025  ... 

## Make FASTA files

In [15]:
# Get information to create the fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

for genotype in genotypes: # ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata[metadata["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata[metadata["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

In [19]:
# print(fasta_files.keys())

In [16]:
# Create fasta files 

# os.chdir(complete_files + "/B3_13_D1_1/" + date_range + "_B3_13_D1_1/")
os.chdir(originals + "complete/")
names = []
for pair in fasta_files.keys():
    # output_path = complete_files + "/B3_13_D1_1/" + date_range + "_B3_13_D1_1/" + pair + "_andersen_updated_" + update_date + ".fasta" 
    output_path = originals + "complete/" + pair + "_andersen_updated_" + update_date + ".fasta"

    output_file = open(output_path, "w")
    for item in fasta_files[pair]:
        # for item in item:
        # item = fasta_files[pair]
        try:
            name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
        except:
            name = str(item[0])
        print(name)
        names.append(name)
        # First is header, second is sequence
        # print(value)
        output_file.write(name + "\n")
        output_file.write(item[1])
    output_file.close()

print(len(names)/8)

>SRR33830355|A/CATTLE/USA/25-016216-005/2025|H5N1|USA|2025|cattle|B3.13
>SRR33830356|A/CATTLE/USA/25-016216-004/2025|H5N1|USA|2025|cattle|B3.13
>SRR33830357|A/CATTLE/USA/25-016216-003/2025|H5N1|USA|2025|cattle|B3.13
>SRR33830358|A/CATTLE/USA/25-016216-002/2025|H5N1|USA|2025|cattle|B3.13
>SRR33830359|A/CATTLE/USA/25-016216-001/2025|H5N1|USA|2025|cattle|B3.13
>SRR33830360|A/CATTLE/USA/25-016215-002/2025|H5N1|USA|2025|cattle|B3.13
>SRR33830362|A/CATTLE/USA/25-016215-001/2025|H5N1|USA|2025|cattle|B3.13
>SRR33830363|A/CATTLE/USA/25-016214-001/2025|H5N1|USA|2025|cattle|B3.13
>SRR33830364|A/CATTLE/USA/25-016212-002/2025|H5N1|USA|2025|cattle|B3.13
>SRR33830365|A/CATTLE/USA/25-016212-001/2025|H5N1|USA|2025|cattle|B3.13
>SRR33830366|A/CATTLE/USA/25-016210-002/2025|H5N1|USA|2025|cattle|B3.13
>SRR33830367|A/CATTLE/USA/25-016210-001/2025|H5N1|USA|2025|cattle|B3.13
>SRR33830368|A/CATTLE/USA/25-016209-001/2025|H5N1|USA|2025|cattle|B3.13
>SRR33830369|A/CATTLE/USA/25-016208-001/2025|H5N1|USA|2025|cattl

## De-Duplication

In [2]:
# De-duplication 

# Gisaid 

gisaid = downloads + "GISAID/complete/B3_13_D1_1/" + date_range + "_B3_13_D1_1_North_America/"

# gisaid = downloads + "Cats/Datasets/GISAID/"

os.chdir(gisaid)

dfs_gisaid = create_dataframes(gisaid)
# dfs_gisaid2 = create_dataframes(gisaid2)

B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2


In [3]:
# Do the same with Andersen 

# dfs_andersen = create_dataframes(complete_files + "/B3_13_D1_1/" + date_range + "_B3_13_D1_1/")
dfs_andersen = create_dataframes(originals + "complete/")

B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2


In [4]:
# os.chdir(downloads)
# dfs_gisaid["B3.13_HA"].to_csv

In [5]:
for key in dfs_andersen.keys():
    dataframes = dfs_andersen[key]
    print(key)

print(dfs_andersen)

B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2
defaultdict(<class 'list'>, {'B3.13_HA': [    isolate_partial                                        full_header  \
0        013627-001  >SRR33565399|A/CATTLE/USA/25-013627-001/2025|H...   
1        013625-001  >SRR33565400|A/CATTLE/USA/25-013625-001/2025|H...   
2        013623-002  >SRR33565401|A/CATTLE/USA/25-013623-002/2025|H...   
3        013623-001  >SRR33565402|A/CATTLE/USA/25-013623-001/2025|H...   
4        013621-005  >SRR33565403|A/CATTLE/USA/25-013621-005/2025|H...   
..              ...                                                ...   
301      015681-003  >SRR33830443|A/CATTLE/USA/25-015681-003/2025|H...   
302      015681-002  >SRR33830444|A/CATTLE/USA/25-015681-002/2025|H...   
303      015681-001  >SRR33830445|A/CATTLE/USA/25-015681-001/2025|H...   
304      036379-001  >SRR33830449|A/CATTLE/USA/24-036379-001-tile/2...   
305 

In [6]:
print(len(list(dfs_gisaid.keys())))
print(len(list(dfs_andersen.keys())))

16
16


In [7]:
# Merge dataframes and drop duplicates

full_dfs = defaultdict(list)
# same = []
# andersen = set()
# gisaid = set()

for i, andersen_key in enumerate(dfs_andersen.keys()):
    if len(dfs_andersen[andersen_key]) > 0:
        for j, gisaid_key in enumerate(dfs_gisaid.keys()):
            if andersen_key == gisaid_key:
                # same.append(gisaid_key)
        # gisaid_key = list(dfs_gisaid.keys())[i]
        # gisaid2_key = list(dfs_gisaid2.keys())[i]

                andersen_df = dfs_andersen[andersen_key][0]
                print(len(andersen_df))
                # print(andersen_df)
                gisaid_df = dfs_gisaid[gisaid_key][0]
                print(len(gisaid_df))
                # gisaid2_df = dfs_gisaid2[gisaid2_key][0]

                print(pd.concat([gisaid_df, andersen_df]).drop_duplicates())

                full_df = pd.concat([andersen_df, gisaid_df], ignore_index=True)
                print("len full df:", len(full_df))
                test = len(full_df.drop_duplicates(subset="isolate_partial"))

                dedup_df = full_df.drop_duplicates(subset="isolate_partial", keep="last")

                # print((full_df.loc[full_df.duplicated(subset="isolate_partial")]))
                # print((full_df.loc[full_df.duplicated(subset="isolate_partial")]))
                # print(full_df)
                
                print("Keeping nothing: ", test)
                
                print("len deduplicated:", len(dedup_df))
                full_dfs[andersen_key].append(dedup_df)
            # else:
                # gisaid.add(gisaid_key)
                # andersen.add(andersen_key)
    
    # break 


# print(full_dfs)
# print(len(full_dfs))
# print(319*8)
# print(len(same))
# print(len(andersen))
# print(len(gisaid))

306
114
    isolate_partial                                        full_header  \
0        015128-004  >EPI_ISL_19890991|A/dairy_cow/USA/015128-004/2...   
1        015124-002  >EPI_ISL_19890990|A/dairy_cow/USA/015124-002/2...   
2        015125-001  >EPI_ISL_19890989|A/dairy_cow/USA/015125-001/2...   
3        015129-001  >EPI_ISL_19890987|A/dairy_cow/USA/015129-001/2...   
4        015129-003  >EPI_ISL_19890986|A/dairy_cow/USA/015129-003/2...   
..              ...                                                ...   
301      015681-003  >SRR33830443|A/CATTLE/USA/25-015681-003/2025|H...   
302      015681-002  >SRR33830444|A/CATTLE/USA/25-015681-002/2025|H...   
303      015681-001  >SRR33830445|A/CATTLE/USA/25-015681-001/2025|H...   
304      036379-001  >SRR33830449|A/CATTLE/USA/24-036379-001-tile/2...   
305      034788-001  >SRR33830450|A/CATTLE/USA/24-034788-001-tile/2...   

                                              sequence  
0    atggagaacatagtactacttcttgcaatagttagccttgt

In [8]:
# If none in one database, only use the other and drop duplicates

full_dfs = defaultdict(list)
for key in dfs_andersen.keys():
    print(key)
# for key in ["D1.3"]:
    dataframes = dfs_andersen[key]
    for i, df in enumerate(dataframes):
        print(i)
        try:
            full_df = df.merge(dfs_gisaid[key][i], how="outer")
            # print(full_df)
            full_df = full_df.drop_duplicates(subset=["isolate_partial"])
            full_dfs[key].append(full_df)
        except:
            print("Failed to merge dataframes in ", key)
            full_dfs[key].append(dataframes[i])

B3.13_HA
0
B3.13_MP
0
B3.13_NA
0
B3.13_NP
0
B3.13_NS
0
B3.13_PA
0
B3.13_PB1
0
B3.13_PB2
0
D1.1_HA
0
D1.1_MP
0
D1.1_NA
0
D1.1_NP
0
D1.1_NS
0
D1.1_PA
0
D1.1_PB1
0
D1.1_PB2
0


## Create FASTA files combining Andersen and GISAID

In [9]:
# Create FASTA files per segment

combined_files = downloads + "Combinations/GISAID_Andersen/" # B3_13_D1_1/" + date_range + "_B3_13_D1_1/"

os.chdir(combined_files)
for pair in full_dfs.keys():
    print(pair)
    output_path = combined_files + pair + "_combined_" + update_date + ".fasta" 

    output_file = open(output_path, "w")
    for item in full_dfs[pair]:
        # for item in item:
        # item = fasta_files[pair]
        for index, row in item.iterrows():
            name = item.loc[index, "full_header"]
            sequence = item.loc[index, "sequence"]
            # print(name)
        # First is header, second is sequence
        # print(value)
            output_file.write(name)
            output_file.write(sequence)
    output_file.close()

B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2
